Importing Libraries

In [1]:
#Importar librerias
import numpy as np
import pandas as pd
import os
import csv
import matplotlib
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import ptitprince as pt
import pyodbc
import openpyxl
import pickle
import itertools
from pandas import to_datetime
from datetime import timedelta, date, datetime
import math
from bayes_opt import BayesianOptimization
import warnings
pd.options.display.max_rows = 999
warnings.filterwarnings("ignore")

Importing and transforming  Data

In [2]:
#carga y procesamiento de los datos
f = r'C:/Users/Anderson Mosquera/universidadean.edu.co/MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes/REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5/Data Tables/HDeviceCGM.txt'
# "C:/Users/Anderson Mosquera/universidadean.edu.co/MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes/REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5/CGM_Editada3.txt"
list_to_append = []
for chunk in pd.read_csv(f,  sep='|', header=0, chunksize=100000):
    list_to_append.append(chunk)
MasterDF = pd.concat(list_to_append)
# MasterDF = pd.read_csv(f, sep='|', header=0, low_memory = False,chunksize=100000 )
# MasterDF = MasterDF[MasterDF.PtID.isin([183, 184,  14, 220, 233,  62,  17, 186,  52, 216, 115,  37, 244,])]
MasterDF = MasterDF[MasterDF['RecordType'] == 'CGM']

In [3]:
#Creating a date time column
MasterDF['Today'] = datetime.today().date()
MasterDF['Date'] = MasterDF['Today'] + pd.to_timedelta(MasterDF['DeviceDtTmDaysFromEnroll'], unit='d')
MasterDF['DeviceTm'] = MasterDF.DeviceTm.astype('str')
MasterDF['DeviceTm'] = MasterDF['DeviceTm'].str[:-2]+ '00'
MasterDF['DateTime'] = MasterDF.Date.astype('str')+ ' '+ MasterDF.DeviceTm
MasterDF['DateTime'] = pd.to_datetime(MasterDF.DateTime, format='%Y-%m-%d %H:%M:%S')
#selecting just the columns for Giammarino's code to run
MasterDF = MasterDF[['PtID','DateTime','GlucoseValue']]
MasterDF = MasterDF.rename(columns={'DateTime':'ts','PtID':'id','GlucoseValue':'gl'})
MasterDF= MasterDF.reset_index(drop=True)
MasterDF = MasterDF.drop_duplicates(subset=['ts','id'])
MasterDF['Dia_Noche'] = MasterDF['ts'].dt.hour.between(7, 18, inclusive='both') \
    .replace({True: 'Dia', False: 'Noche'})
    # MasterDF = MasterDF[MasterDF['Dia_Noche']=='Dia']
MasterDF=MasterDF[['ts', 'id','gl']]
data = MasterDF

In [4]:

def Event(Data,glucose_threshold):
    'Data has to be type list'
    'Threshold should be an integer'
    C1 = 0
    C2 = 0
    for i in range(len(Data)) :
        if Data[i] < glucose_threshold:
            C1+=1
        if C1 == 3:
            C2+=1
        elif C1 >2:
            pass
        else:
            pass
    else:
        C1=0
    if C2 > 1:
        return 1
    else:
        return 0

# create a list for storing the data

def New_Sequences(data):
    sequences = []
    NotWorking = []
    minutes = 5
    Days_Week = 7 # this is the number of days to be considered i na week, can be changed
    glucose_threshold = 54
    # calculate the number of timestamps in one week
    sequence_length = int(Days_Week * 24 * 60 // minutes)
    for patient in data.id.unique():
        # Data = pd.DataFrame()
        if type(patient) != 0:
            
            try:
                #Do preprocessing inside the function per patient
                Data = data[data['id'].isin([patient])]
                Data = data[data['gl'].between(40, 400)]
                # reshape the dataset from long to wide
                Data = Data.pivot(index='ts', columns=['id'], values=['gl'])
                Data.columns = Data.columns.get_level_values(level='id')
                Data.reset_index(inplace = True)
                Data['date'] = pd.to_datetime(Data['ts']).dt.date
                Data=Data[Data[patient].notnull()]
                #Per Patient
                min_Date = datetime.strptime(str(Data['ts'].min())[:-9], '%Y-%m-%d').date()
                # print(min_Date)
                max_Date = datetime.strptime(str(Data['ts'].max())[:-9], '%Y-%m-%d').date()
                # print(max_Date)
                Difference = abs(max_Date-min_Date).days #difference in days between the two dates  
                Sequences = math.ceil(Difference/Days_Week) # Define the number of 1 week sequences
                Result = pd.DataFrame()
                for i in range (1, Sequences, 1):
                    # generate the range
                    date_generated = pd.DataFrame()    
                    date_generated = [min_Date + timedelta(days=x) for x in range(0, (timedelta(days=Days_Week)).days)]
                    # print(len(date_generated))
                    min_Date = min_Date + timedelta(days=Days_Week+1)
                    df_Generated = pd.DataFrame(date_generated)
                    df_Generated = df_Generated.rename(columns={0:'date'})
                    df_Generated = pd.merge(df_Generated,Data,'left',left_on='date',right_on='date')
                    df_Generated['Sequence'] = i
                    Result = pd.concat([Result,df_Generated])
                Grouped = Result.groupby('Sequence').count()
                Grouped['Total_Readings_Sequence'] = sequence_length
                Grouped = Grouped[['ts','Total_Readings_Sequence']]
                Grouped['Time_Worn'] = Grouped['ts']/Grouped['Total_Readings_Sequence']
                Grouped['Criteria'] = np.where(Grouped['Time_Worn']>0.7, "Applicable", 'Not Applicable')
                Grouped = Grouped[Grouped['Criteria']=='Applicable']
                Result = Result[Result['Sequence'].isin(Grouped.index.to_series())]          
                for i in Grouped.index.to_series():
                    #Evaluate if next is applicable
                    try:
                        if Grouped.loc[i+1]['Criteria'] == 'Applicable':
                            X = Result[patient][Result['Sequence'] == i].dropna().to_list()
                            L = len(X)
                            Y = Result[patient][Result['Sequence'] == i+1].to_list()
                            Y = Event(Y, glucose_threshold)
                            #I need to determine the amount of consecutive data below the threshold and if it greater that 15 minutes (three readings) then Y = 1
                            # save the patient's data
                            sequences.append({
                            'patient': patient,
                            'Sequence': i,
                            'start': str(Result['ts'][Result['Sequence'] == i].min()),
                            'end': str(Result['ts'][Result['Sequence'] == i].max()),
                            'L': L,
                            'X': X,
                            'Y': Y
                            })
                        else:
                            pass
                    except:
                        pass
            except:
                NotWorking.append(patient)
        else:
            pass
    return sequences

In [5]:
sequences = New_Sequences(data)

In [6]:
#Balancing the Dataset
import random
positives = 0
sequences_bal = []
sequences_neg = []
for i in range(len(sequences)):
    if sequences[i]['Y'] == 1:
        positives += 1
        sequences_bal.append(sequences[i])
    else:
        sequences_neg.append(sequences[i])
positives
random.shuffle(sequences_neg)
len(sequences_neg[:positives])
sequences_bal.extend(sequences_neg[:positives])
len(sequences_bal)
sequences = sequences_bal

In [7]:
len(sequences)

2112

Creating a BlackBox Function to optimize

In [16]:
def Run_Models(sequences, l1_penalty,l2_penalty, learning_rate, batch_size,  skf, Model):

    model = Model
    results = []

    # loop across the folds
    for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
        
        # fit the model to the training set


        model.fit(
            sequences=[sequences[i] for i in train_index],
            sequence_length=int(7 * 24 * 60 // 5),
            l1_penalty=l1_penalty,
            l2_penalty=l2_penalty,
            learning_rate=learning_rate,
            batch_size=batch_size,
            epochs=1000,
            seed=42,
            verbose=0
        )

        # evaluate the model on the test set
        metrics = model.evaluate(sequences=[sequences[i] for i in test_index])

        # save the results
        results.append(metrics)

    # organize the results in a data frame
    results = pd.DataFrame(results)

    return results.mean().values[6]

In [17]:
# to run the experiments
from src.model import Model
from sklearn.model_selection import StratifiedKFold
model = Model()
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

Optimizing Parameters via Bayesian Optimization

In [43]:
def black_box_function(l1_penalty,l2_penalty, learning_rate): #
    global sequences
    global skf
    global model
    batch_size = 16
    return Run_Models(sequences, l1_penalty,l2_penalty, learning_rate, batch_size,  skf, model)

In [44]:
pbounds = {"l1_penalty": [0.001, 1] , "l2_penalty": [0.001, 1], 'learning_rate': [0.000001, 1] }


In [45]:
optimizer = BayesianOptimization(f = black_box_function,
                                pbounds = pbounds, verbose = 2,
                                random_state = 4)
optimizer.maximize(init_points = 1, n_iter = 2)
print("Best result: {}; f(x) = {}.".format(optimizer.max["params"], optimizer.max["target"]))

|   iter    |  target   | l1_pen... | l2_pen... | learni... |
-------------------------------------------------------------


KeyboardInterrupt: 

Optimizing Paramters via Randomized Grid Search

In [ ]:
#Defining Hyper Parameters
l1_penalty=[random.uniform(0.001, 0.1), random.uniform(0.001, 0.1), random.uniform(0.001, 0.1)]
l2_penalty=[random.uniform(0.001, 0.1), random.uniform(0.001, 0.1), random.uniform(0.001, 0.1)]
learning_rate = [random.uniform(0.000001, 0.1), random.uniform(0.000001, 0.1), random.uniform(0.000001, 0.1)]
batch_size = 16

Parameters = []
# crear una matriz con las combinaciones de parametros
for element in itertools.product(l1_penalty,l2_penalty,learning_rate,batch_size):
    Parameters.append(list(element))
len(Parameters)

In [ ]:
Random_Search_Results = pd.DataFrame(columns=['Parameters','AUC'])
for i in [x for x in range (len(Parameters))]:
    l1_penalty = Parameters[i]
    l2_penalty = Parameters[i]
    learning_rate = Parameters[i]
    batch_size = Parameters[i]
    try:
        R = Run_Models(sequences, l1_penalty,l2_penalty,learning_rate,batch_size, skf, Model)
        
        Random_Search_Results = Random_Search_Results.append(
            {'Parameters': str(Parameters[i]) , 'AUC': R }, 
            ignore_index=True)
    except:
        continue
Random_Search_Results.sort_values(by='AUC', ascending=False)

Optimizating Parameters vis Grid Search

In [ ]:
#Defining Hyper Parameters
l1_penalty=[0.001, 0.01, 0.1]
l2_penalty=[0.001, 0.01, 0.1]
learning_rate = [0.000001, 0.0001, 0.1]
batch_size = 16

Parameters = []
# crear una matriz con las combinaciones de parametros
for element in itertools.product(l1_penalty,l2_penalty,learning_rate,batch_size):
    Parameters.append(list(element))
len(Parameters)

In [ ]:
Grid_Search_Results = pd.DataFrame(columns=['Parameters','AUC'])
for i in [x for x in range (len(Parameters))]:
    l1_penalty = Parameters[i]
    l2_penalty = Parameters[i]
    learning_rate = Parameters[i]
    batch_size = Parameters[i]
    try:
        R = Run_Models(sequences, l1_penalty,l2_penalty,learning_rate,batch_size, skf, Model)
        
        Grid_Search_Results = Grid_Search_Results.append(
            {'Parameters': str(Parameters[i]) , 'AUC': R }, 
            ignore_index=True)
    except:
        continue
Grid_Search_Results.sort_values(by='AUC', ascending=False)